# Part 2 — TESSERA pilot (25 patches) and RF pilot on the same patches

**Goal:**
1. download TESSERA v1.1 (2019) for the 25 patches of the RF pilot, on the PASTIS grid (integer crop, no resampling), with an alignment check per patch;
2. run `baselines/rf/train_batch.py` unchanged on these embeddings, to compare with its results on the bilinear version (same patches, same model, same seed: only the alignment differs).

**Needs Part 1 on the Drive:** `pj_vmi/processed/` and `pj_vmi/pastis_subset/`. **Runtime:** CPU. **Time:** about 10–15 min. **Drive space:** about 60 MB.

**Files needed in the repository** (new in this part): `src/data/align_qa.py`, `src/data/shards.py`, `src/data/embeddings.py`, `scripts/fetch_tessera.py`, `scripts/export_rf_layout.py`, `scripts/compare_alignment.py`, `notebooks/data/02_tessera_pilot.ipynb`, and the updated `src/data/pastis_grid.py`.

## 1. Runtime, Drive and configuration

In [ ]:
import sys, shutil, platform
from pathlib import Path
import json, numpy as np, pandas as pd
print("Python", platform.python_version())
assert sys.version_info >= (3, 12), "geotessera needs Python >= 3.12"

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"
CODE = Path("/content/code")

PROC = Path("/content/processed")
PROC_DRIVE = DRIVE / "processed"
SUBSET = DRIVE / "pastis_subset"
PILOT = DRIVE / "pilot"                 # small, kept on the Drive
RF_ROOT = Path("/content/rf_layout")    # ROOT given to the RF pilot scripts

## 2. Code

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt

## 3. Part 1 outputs
Copies the small files of `processed/` from the Drive to the local disk.

**Done when:** 2,433 patches and 25 patches in the selection.

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "labels.npy", "instances.npy", "parcel_ids.npy", "splits.json", "dates_s2.json",
          "selection_seed42.json"]:
    shutil.copy(PROC_DRIVE / f, PROC / f)
meta = pd.read_csv(PROC / "meta.csv", index_col="idx")
sel = json.loads((PROC / "selection_seed42.json").read_text())
PILOT_IDS = [p["patch_id"] for p in sel["patches"]]
print(len(meta), "patches,", len(PILOT_IDS), "in the selection")
assert (SUBSET / "PASTIS" / "metadata.geojson").exists(), "run Part 1 first (pastis_subset missing)"

## 4. TESSERA pilot
`fetch_tessera.py` reads TESSERA through the geotessera Zarr API and stops if the dataset is not `1.1-dclimate`: the v1.1 variants cannot be mixed. Each patch is an integer crop of the stored int8 embedding (`q`, plus a per-pixel `scale`). The first read can take a few minutes (store initialisation).

Alignment check: inside parcels, pixels of one parcel should look alike, so the within-parcel share of variance (`ratio`) should be lowest with no shift. The script also tries shifts of ±1 pixel and records the best one.

**Done when:**
- `TESSERA dataset: 1.1-dclimate`;
- `errors: 0`;
- in the table: empty `error`, `nodata_frac` = 0, `best_dx` = `best_dy` = 0 for (almost) all patches, `ratio_+0_+0` around 0.2–0.4 (0.20–0.36 on 4 test patches on 26/10).

If 2 or more patches have a best shift other than (0, 0), stop and report the table.

In [ ]:
ids = " ".join(map(str, PILOT_IDS))
!python scripts/fetch_tessera.py --processed "{PROC}" --year 2019 --shard-dir "{PILOT}/tessera_2019" --ids {ids}

In [ ]:
d = np.load(PILOT / "tessera_2019" / "pilot_0000.npz")
qa = pd.DataFrame({k[3:]: d[k] for k in d.files if k.startswith("qa_")})
qa.insert(0, "ID_PATCH", meta.loc[d["idx"], "ID_PATCH"].values)
pd.set_option("display.width", 200)
print(qa[["ID_PATCH", "error", "nodata_frac", "best_dx", "best_dy", "ratio_+0_+0", "seconds"]].to_string(index=False))
aligned = ((qa.best_dx == 0) & (qa.best_dy == 0)).sum()
print(f"\nbest shift (0, 0): {aligned}/{len(qa)}")
print(f"estimated time for 2,433 patches with 6 workers: {qa.seconds.median() * 2433 / 6 / 60:.0f} min")
print(json.loads((PILOT / "tessera_2019" / "info.json").read_text()))

## 5. RF pilot on these embeddings
`export_rf_layout.py` writes the pilot embeddings in the folder layout of the RF pilot scripts:
`PASTIS/` (labels, parcel ids and S2 of the 25 patches, from `pastis_subset`) and `TESSERA/batch_2019/patch_<id>/aligned.tif` + `selection.json`.
`train_batch.py` then runs **unchanged**: only its `ROOT` line is replaced, in a copy outside the repository.

**Done when:** the script prints accuracy and macro-F1 for val and test, and writes `baseline_results.json`. Put these numbers next to those of the bilinear version (`baseline_results.json` of the RF pilot).

In [ ]:
!python scripts/export_rf_layout.py --pastis "{SUBSET}" --processed "{PROC}" --name tessera_2019 \
    --shard "{PILOT}/tessera_2019/pilot_0000.npz" --root "{RF_ROOT}"

In [ ]:
RUN = Path("/content/rf_run")
RUN.mkdir(exist_ok=True)
OLD_ROOT = 'ROOT = Path(r"G:\\Vmi\\project")'
src = (CODE / "baselines" / "rf" / "train_batch.py").read_text(encoding="utf-8")
assert OLD_ROOT in src, "the ROOT line of train_batch.py has changed: update OLD_ROOT"
(RUN / "train_batch.py").write_text(src.replace(OLD_ROOT, f'ROOT = Path(r"{RF_ROOT}")'), encoding="utf-8")
!cd "{RUN}" && python train_batch.py

In [ ]:
res_file = RF_ROOT / "TESSERA" / "batch_2019" / "baseline_results.json"
res = json.loads(res_file.read_text())
for split in ["val", "test"]:
    print(f"{split}: accuracy {res[split]['accuracy']:.4f}, macro-F1 {res[split]['macro_f1']:.4f}, "
          f"majority {res[split]['majority_accuracy']:.4f}, unseen classes {res[split]['unseen_classes']}")
shutil.copy(res_file, PILOT / "rf_baseline_results_integer_crop.json")

## 6. Direct comparison with the bilinear version (optional)
Needs the RF pilot's own `aligned.tif` files on the Drive: `pj_vmi/rf_pilot/batch_2019/selection.json` and `pj_vmi/rf_pilot/batch_2019/patch_<id>/aligned.tif` (the `raw/` folders are not needed).

**Done when:** every patch peaks at shift (0, 0), and `origin_dx_m` / `origin_dy_m` equal minus the snap correction of its tile (e.g. t30uxv: +2.36 / +0.62).

In [ ]:
RF_OWN = DRIVE / "rf_pilot" / "batch_2019"
if RF_OWN.exists():
    !python scripts/compare_alignment.py --a "{RF_ROOT}/TESSERA/batch_2019" --b "{RF_OWN}" --out "{PILOT}/compare_alignment.csv"
    print(meta.groupby("TILE")[["snap_dx", "snap_dy"]].first().round(2))
else:
    print("not on the Drive yet:", RF_OWN)

## Results to report
1. The commit line of step 2.
2. The full output of the fetch cell and the table of step 4.
3. The accuracy / macro-F1 lines of step 5.
4. The output of step 6 if it ran.
5. Any error message, in full.